In [ ]:
# -*- coding: utf-8 -*-
"""
Robust soybean rough classifier (September 2025 intent) using Microsoft Planetary Computer Sentinel-2 L2A + USDA CDL.
This version is resilient to multiple stackstac output shapes (band coord missing / numeric / string / bands as variables).
Outputs: feature stack GeoTIFF, CDL GeoTIFF, soy mask GeoTIFF, classification GeoTIFF, and validation report.
"""
import os
import sys
from datetime import datetime
import re
import numpy as np
import xarray as xr
import rioxarray
from shapely.geometry import box, mapping
from pystac_client import Client
import planetary_computer as pc
import stackstac
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# -----------------------
# USER PARAMETERS (edytuj)
# -----------------------
OUTDIR = "outputs_soy_2025_robust"
os.makedirs(OUTDIR, exist_ok=True)

# AOI: wpisz bbox (minx, miny, maxx, maxy)
AOI = (-94.5, 41.5, -93.0, 42.5)  # przykładowe Iowa -> zamień na swoje bbox lub podaj większy region

# okres: sezon wiosna-wrzesień 2025 (jeśli chcesz inny okres -> zmień)
START_DATE = "2025-04-01"
END_DATE   = "2025-09-30"

# Sentinel-2 bands we try to use (prioritized)
S2_BANDS = ["B02","B03","B04","B08","B11","B12"]  # blue, green, red, nir, swir1, swir2

# pamięciooszczędne ustawienia
RESOLUTION = 20        # 20 m (change to 10 if masz RAM)
LIMIT_ITEMS = 40       # max scenes to request; zmniejsz żeby odciążyć
CLOUD_COVER_MAX = 35   # filtr chmur (procent)

# CDL preferences
WANTED_CDL_YEAR = 2025
SOY_CODES = [5]        # typowy kod soi w CDL; sprawdź legendę dla pewności

# klasyfikator
RANDOM_STATE = 42
N_ESTIMATORS = 150
N_SAMPLES_PER_CLASS = 2000
TEST_SIZE = 0.2

OUT_EPSG = 3857

# -----------------------
# Helper functions
# -----------------------
def bbox_to_geojson(bbox):
    minx,miny,maxx,maxy = bbox
    return mapping(box(minx, miny, maxx, maxy))

def find_band_like_vars(ds):
    """Return list of data variable names that look like Sentinel band names (B02, B03, B8, '02' etc.)."""
    candidates = []
    for name in ds.data_vars.keys():
        n = str(name)
        if re.match(r"(?i)^b0?\d$|(?i)^b1[0-2]$|^B\d{1,2}$", n) or re.match(r"^\d{1,2}$", n):
            candidates.append(n)
    return candidates

def ensure_band_coord(ds, prefer_bands=None):
    """
    Ensure dataset has a 'band' coordinate with names like 'B02','B04' etc.
    Handles cases:
      - ds.coords['band'] exists (numeric or string): normalize to 'Bxx'
      - ds has data variables per-band e.g. ds['B02'], ds['B04'] -> concat into band dim
      - fallback: try to find any variable names that match band-like pattern
    Returns dataset with dims ('time','band','y','x') or ('band','y','x') for single-time.
    """
    # 1) if 'band' coord exists and non-empty: normalize its values to "Bxx"
    if 'band' in ds.coords and len(ds.coords['band'].values) > 0:
        raw = list(ds.coords['band'].values)
        # map numeric -> Bxx
        mapped = []
        for v in raw:
            try:
                if isinstance(v, (np.integer, int, np.floating, float)):
                    vn = int(np.round(float(v)))
                    mapped.append(f"B{vn:02d}")
                else:
                    s = str(v).strip()
                    if s.upper().startswith("B"):
                        mapped.append(s.upper())
                    else:
                        # try numeric string
                        try:
                            vn = int(float(s))
                            mapped.append(f"B{vn:02d}")
                        except Exception:
                            mapped.append(s.upper())
            except Exception:
                mapped.append(str(v))
        ds = ds.assign_coords(band=("band", mapped))
        return ds

    # 2) if bands are present as data variables (common case)
    available = list(ds.data_vars.keys())
    # prefer explicit S2_BANDS order if present
    ordered = []
    for b in (prefer_bands or S2_BANDS):
        if b in available:
            ordered.append(b)
    # if none of the prefer_bands present, try to find band-like variables
    if len(ordered) == 0:
        candidates = find_band_like_vars(ds)
        # normalize names like '02' -> 'B02'
        cand_norm = []
        for c in candidates:
            s = str(c)
            if s.upper().startswith("B"):
                cand_norm.append(s.upper())
            else:
                try:
                    vn = int(float(s))
                    cand_norm.append(f"B{vn:02d}")
                except Exception:
                    cand_norm.append(s)
        # match back to variables and keep both original var name and assignment name
        ordered = []
        mapping_var_to_band = {}
        for orig, norm in zip(candidates, cand_norm):
            ordered.append(orig)  # keep original var name to extract data
            mapping_var_to_band[orig] = norm

        if len(ordered) == 0:
            # give up - cannot find bands
            raise ValueError("Nie mogę znaleźć pasm Sentinel-2 w zwróconym DS. Zawartość ds.data_vars: " + ", ".join(available))
        # Build DataArray list from discovered variables
        arrays = []
        band_names = []
        for var in ordered:
            da = ds[var]
            # ensure dims order y,x or time,y,x
            if 'time' in da.dims and da.sizes.get('time',1) > 1:
                # take mean over time if multi-temporal var unexpectedly present
                da = da.mean(dim='time')
            da = da.squeeze()  # remove singletons
            # we expect dims now (y,x)
            arrays.append(da)
            band_names.append(mapping_var_to_band.get(var, str(var)))
        # concat into band dim
        ds_band = xr.concat(arrays, dim='band')
        ds_band = ds_band.assign_coords(band=band_names)
        # ensure y,x coords preserved
        return ds_band

    # 3) else: if ordered contains band names equal to data_vars, build concat
    arrays = []
    band_names = []
    for var in ordered:
        da = ds[var]
        da = da.squeeze()
        arrays.append(da)
        band_names.append(var)
    ds_band = xr.concat(arrays, dim='band')
    ds_band = ds_band.assign_coords(band=band_names)
    return ds_band

def compute_ndvi_from_med(ds_med):
    # expects ds_med with band coord names like 'B04','B08' or variables named accordingly
    bands = list(ds_med.coords['band'].values)
    if isinstance(bands[0], bytes):
        bands = [b.decode() for b in bands]
    bands = [str(b) for b in bands]
    if "B08" not in bands or "B04" not in bands:
        # try to detect numeric equivalents 8 and 4
        if "8" in bands and "4" in bands:
            b08 = "8"; b04 = "4"
        else:
            raise ValueError("Brakuje pasm B04 lub B08 do obliczenia NDVI. Znalezione bandy: " + ",".join(bands))
        nir = ds_med.sel(band=b08).squeeze("band", drop=True)
        red = ds_med.sel(band=b04).squeeze("band", drop=True)
    else:
        nir = ds_med.sel(band="B08").squeeze("band", drop=True)
        red = ds_med.sel(band="B04").squeeze("band", drop=True)
    ndvi = (nir - red) / (nir + red + 1e-6)
    return ndvi

# -----------------------
# STAC client / CDL discovery
# -----------------------
stac_url = "https://planetarycomputer.microsoft.com/api/stac/v1"
pc_client = Client.open(stac_url)

print("Searching for available USDA CDL years on Planetary Computer (AOI)...")
cdl_search = pc_client.search(collections=["usda-cdl"], intersects=bbox_to_geojson(AOI), limit=200)
cdl_items = list(cdl_search.get_items())
available_years = set()
for it in cdl_items:
    props = it.properties
    y = props.get("year")
    if y:
        try:
            available_years.add(int(y))
        except Exception:
            pass

if not available_years:
    # fallback: try without AOI
    print("Nie znaleziono CDL w AOI — przeszukuję kolekcję bez AOI (fallback)...")
    cdl_search = pc_client.search(collections=["usda-cdl"], limit=500)
    cdl_items = list(cdl_search.get_items())
    for it in cdl_items:
        y = it.properties.get("year")
        if y:
            try:
                available_years.add(int(y))
            except Exception:
                pass

if not available_years:
    raise SystemExit("Brak elementów CDL w Planetary Computer — nie można kontynuować.")

latest_cdl = max(available_years)
if WANTED_CDL_YEAR in available_years:
    chosen_cdl_year = WANTED_CDL_YEAR
    print(f"CDL {WANTED_CDL_YEAR} dostępny — użyję go.")
else:
    chosen_cdl_year = latest_cdl
    print(f"CDL {WANTED_CDL_YEAR} niedostępny. Użyję najnowszego dostępnego roku: {chosen_cdl_year}.")

# -----------------------
# get Sentinel-2 items & stack
# -----------------------
print(f"Wyszukiwanie Sentinel-2 L2A dla okresu {START_DATE} - {END_DATE} (AOI)...")
search = pc_client.search(
    collections=["sentinel-2-l2a"],
    intersects=bbox_to_geojson(AOI),
    datetime=f"{START_DATE}/{END_DATE}",
    query={"eo:cloud_cover": {"lt": CLOUD_COVER_MAX}},
    limit=LIMIT_ITEMS
)
items = list(search.get_items())
if len(items) == 0:
    raise SystemExit("Brak scen Sentinel-2 dla podanego zakresu/daty/AOI.")

signed_items = [pc.sign(it) for it in items]
print(f"Znaleziono {len(signed_items)} scen (signed). Stackuję...")

ds_raw = stackstac.stack(
    signed_items,
    assets=S2_BANDS,
    resolution=RESOLUTION,
    epsg=OUT_EPSG,
    resampling="bilinear",
    chunksize=2048,
    bounds=AOI
)
print("stackstac zwrócił dataset. Sprawdźmy strukturę...")

# Print a short summary for diagnostics
print("ds_raw dims:", ds_raw.dims)
print("ds_raw coords:", {k: list(ds_raw.coords[k].values)[:6] for k in ds_raw.coords.keys() if k in ['band','time']})

# Normalize / ensure band coordinate
try:
    ds_for_med = ensure_band_coord(ds_raw)
except Exception as e:
    # Jeśli ensure_band_coord się nie uda, spróbuj jeszcze raz z alternatywą:
    print("ensure_band_coord zgłosił błąd:", e)
    # jeśli ds_raw zawiera pod-namespace 'bands' lub inaczej, spróbujemy heurystyką
    # Final fallback: spróbuj wybrać pierwszą warstwę time i użyć jej zmiennych
    # (rzadkie scenariusze)
    raise

# Jeżeli ds_for_med ma wymiar 'time' -> policz medianę, jeśli nie ma -> traktuj jako pojedyncze obrazy
if 'time' in ds_for_med.dims and ds_for_med.sizes.get('time', 0) > 1:
    print("Mamy wymiar time -> obliczam medianę po czasie (skipna=True).")
    ds_med = ds_for_med.median(dim="time", skipna=True)
else:
    print("Brak wymiaru time (single-scene) -> używam bez redukcji.")
    # upewnij się, że ds_for_med ma wymiar band,y,x (jeśli ma tylko y,x to zamienimy)
    ds_med = ds_for_med

# compute NDVI robustly
print("Obliczam NDVI z medianowego obrazu...")
ndvi_med = compute_ndvi_from_med(ds_med)

# Build feature dataset: band_median for available bands + NDVI
feat_vars = {}
bands_present = [str(b) for b in ds_med.coords['band'].values]
for b in S2_BANDS:
    if b in bands_present:
        feat_vars[f"{b}_med"] = ds_med.sel(band=b).squeeze("band", drop=True)
    else:
        print(f"Uwaga: pasmo {b} nieobecne w medianie i zostanie pominięte.")
feat_vars["NDVI_med"] = ndvi_med.squeeze("band", drop=True)

feat_ds = xr.Dataset(feat_vars)
feat_ds.rio.write_crs(f"EPSG:{OUT_EPSG}", inplace=True)
feat_tif = os.path.join(OUTDIR, "sentinel_median_features_robust.tif")
print("Zapisuję feature stack:", feat_tif)
feat_ds.rio.to_raster(feat_tif, compress="LZW")

# -----------------------
# Pobierz CDL i przygotuj maskę soi
# -----------------------
print(f"Szukam CDL dla roku {chosen_cdl_year} w AOI...")
cdl_search = pc_client.search(collections=["usda-cdl"], intersects=bbox_to_geojson(AOI), query={"year": {"eq": chosen_cdl_year}}, limit=50)
cdl_items = list(cdl_search.get_items())
if len(cdl_items) == 0:
    print("Nie znaleziono CDL filtrowanego po roku/AOI — używam pierwszego itemu kolekcji (fallback).")
    cdl_search_all = pc_client.search(collections=["usda-cdl"], limit=200)
    cdl_items = list(cdl_search_all.get_items())
    if len(cdl_items) == 0:
        raise SystemExit("Brak elementów CDL w Planetary Computer (fallback).")
cdl_item = pc.sign(cdl_items[0])
asset_keys = list(cdl_item.assets.keys())
asset_name = None
for k in asset_keys:
    if "cdl" in k.lower() or "data" in k.lower() or "raster" in k.lower():
        asset_name = k; break
if asset_name is None:
    asset_name = asset_keys[0]
cdl_href = cdl_item.assets[asset_name].href
print("CDL href:", cdl_href)

cdl = rioxarray.open_rasterio(cdl_href, masked=True).squeeze("band", drop=True)
cdl_matched = cdl.rio.reproject_match(feat_ds)

soy_mask = (np.isin(cdl_matched.values, SOY_CODES)).astype(np.uint8)
soy_da = xr.DataArray(soy_mask, coords={"y": cdl_matched.y, "x": cdl_matched.x}, dims=("y","x"))
soy_da.rio.write_crs(cdl_matched.rio.crs, inplace=True)

cdl_tif = os.path.join(OUTDIR, f"cdl_{chosen_cdl_year}_robust.tif")
soy_tif = os.path.join(OUTDIR, f"soy_mask_{chosen_cdl_year}_robust.tif")
print("Zapisuję CDL i soy mask:", cdl_tif, soy_tif)
cdl_matched.rio.to_raster(cdl_tif, compress="LZW")
soy_da.rio.to_raster(soy_tif, compress="LZW")

# -----------------------
# Przygotuj próbki (balanced) i trenuj
# -----------------------
feat_names = list(feat_ds.data_vars.keys())
F = np.stack([feat_ds[v].values for v in feat_names], axis=-1)  # (y,x,feat)
Y = soy_da.values  # (y,x)

valid = ~np.isnan(F).any(axis=-1) & (~np.isnan(Y))
ys = Y[valid]
fs = F[valid]
print("Dostępnych oznaczonych pikseli (valid):", fs.shape[0])
idx_soy = np.where(ys==1)[0]
idx_other = np.where(ys==0)[0]
n_possible = min(len(idx_soy), len(idx_other))
n_take = min(N_SAMPLES_PER_CLASS, n_possible)
if n_take < 10:
    raise SystemExit("Za mało oznaczonych pikseli dla obu klas. Zwiększ AOI lub wybierz inny rok CDL.")

rng = np.random.RandomState(RANDOM_STATE)
s_soy = rng.choice(idx_soy, size=n_take, replace=False)
s_other = rng.choice(idx_other, size=n_take, replace=False)
sel = np.concatenate([s_soy, s_other])
X = fs[sel]
y = ys[sel]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
print(f"Trenuję RandomForest na {X_train.shape[0]} próbkach...")
clf = RandomForestClassifier(n_estimators=N_ESTIMATORS, n_jobs=-1, random_state=RANDOM_STATE)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print("Wyniki walidacji:")
print(classification_report(y_test, y_pred, target_names=["other","soybean"]))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# -----------------------
# Predykcja na całym rastrze (lekka)
# -----------------------
h,w,nf = F.shape
F2 = F.reshape(-1, nf)
valid_pixels = ~np.isnan(F2).any(axis=1)
pred = np.full((h*w,), 255, dtype=np.uint8)
print("Predykcja dla", valid_pixels.sum(), "pikseli...")
pred[valid_pixels] = clf.predict(F2[valid_pixels])
pred_raster = pred.reshape(h, w)

pred_da = xr.DataArray(pred_raster, coords={"y": feat_ds.y, "x": feat_ds.x}, dims=("y","x"))
pred_da.rio.write_crs(feat_ds.rio.crs, inplace=True)
pred_tif = os.path.join(OUTDIR, f"soy_class_robust_{START_DATE}_to_{END_DATE}.tif")
print("Zapisuję mapę klasyfikacji:", pred_tif)
pred_da.rio.to_raster(pred_tif, compress="LZW")

print("Zakończono. Wyniki w katalogu:", OUTDIR)
